In [ ]:
# Colab setup: mount Drive and move into the project folder (edit PROJECT if your folder is elsewhere)
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = "/content/drive/MyDrive/Colab Notebooks/CapstoneProject/careloop-no-show-guard"
    os.chdir(PROJECT)
except ImportError:
    pass                                   # not in Colab: stay in the current folder
print(os.getcwd()); print(os.listdir("."))

# 04 – GenAI: structured reasons from call-centre notes (Part 4)
**Never paste your key into a cell.** It is typed at run time with `input()`; clear the output and Edit → Find for the first characters of the key before downloading.

In [ ]:
!pip install -q google-genai pydantic

In [ ]:
import json, time, pandas as pd
from google import genai
from google.genai import types
from pydantic import BaseModel, Field, ValidationError

MODEL_NAME = "PUT-THE-FLASH-MODEL-NAME-FROM-SESSION-22.1-HERE"
API_KEY = input("Paste your key: ")
client = genai.Client(api_key=API_KEY)
CALLS = 0   # running count of API calls

ALLOWED = ["forgot_or_no_reminder","distance_or_transport","cost_or_fee","illness_or_emergency","work_or_schedule","other"]
SCHEMA = {"type": "object",
          "properties": {"root_cause": {"type": "string"}, "confidence": {"type": "number"},
                         "summary": {"type": "string"}, "recommended_action": {"type": "string"}},
          "required": ["root_cause", "confidence", "summary", "recommended_action"]}

SYSTEM_TEXT = ("You are a careful operations analyst for CareLoop Clinics. You read short call-centre notes "
               "and reply with JSON only, matching the requested schema.")

def craft_prompt(note_text):
    return f"""CONTEXT: CareLoop Clinics runs outpatient clinics in India. A call-centre agent phoned a patient who missed an appointment and wrote the note below. The note may be in English or in Hinglish (Hindi written in English letters).
ROLE: You are a healthcare operations analyst.
ACTION: Identify the single main reason the patient missed the appointment. root_cause must be exactly one of: {", ".join(ALLOWED)}. If the note is ambiguous or no listed cause clearly fits, answer "other" rather than guess. Also give confidence (0 to 1), a summary of at most 25 words, and one recommended_action for the clinic.
TONE: Neutral, concise, professional.
FORMAT: JSON with exactly the fields root_cause, confidence, summary, recommended_action.

NOTE: {note_text}"""

def call_model(prompt):
    """One API call; returns the raw JSON text exactly as returned."""
    global CALLS
    CALLS += 1
    response = client.models.generate_content(
        model=MODEL_NAME, contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT, temperature=0, max_output_tokens=2000,
            response_mime_type="application/json", response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(thinking_level="minimal")))   # delete this line if your model rejects it
    return response.text

LAST_RAW = ""
def analyse_note(note_text):
    """Send one note; returns the parsed dict. The raw reply text stays in LAST_RAW."""
    global LAST_RAW
    LAST_RAW = call_model(craft_prompt(note_text))
    return json.loads(LAST_RAW)

## Task 4.1 – one validated call (N001)

In [ ]:
notes = pd.read_csv("data/followup_notes.csv")
first = notes.iloc[0]
print(first["note_id"], "|", first["note_text"])
print(analyse_note(first["note_text"]))

## Task 4.2 – validate and process the first 10 notes

In [ ]:
class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

def check_allowed(value):
    if value not in ALLOWED:
        raise ValueError(f"root_cause '{value}' not in allowed list")

results = []
for _, row in notes.head(10).iterrows():
    note_text = row["note_text"]
    rec = {"note_id": row["note_id"], "appointment_id": row["appointment_id"], "root_cause": "", "confidence": "",
           "summary": "", "recommended_action": "", "validation_status": "", "raw_response": ""}
    if pd.isna(note_text) or str(note_text).strip() == "":
        rec.update(root_cause="other", confidence=0.0, summary="No note recorded",
                   recommended_action="Re-call patient", validation_status="skipped_blank")
        results.append(rec); continue
    LAST_RAW = ""
    try:
        try:
            data = analyse_note(note_text)               # Task 4.2 says: call analyse_note
        finally:
            rec["raw_response"] = LAST_RAW               # keep the reply exactly as returned
        for k in ["root_cause", "confidence", "summary", "recommended_action"]:   # copy whatever fields exist, never repair
            if isinstance(data, dict) and k in data:
                rec[k] = data[k]
        insight = FollowupInsight(**data)
        check_allowed(insight.root_cause)
        rec["validation_status"] = "valid"
    except Exception as e:
        rec["validation_status"] = "invalid"
        print(row["note_id"], "invalid:", type(e).__name__, str(e)[:120])
    results.append(rec)
    time.sleep(13)                                  # free-tier rate limit

analysed = pd.DataFrame(results, columns=["note_id","appointment_id","root_cause","confidence","summary",
                                          "recommended_action","validation_status","raw_response"])
analysed.to_csv("notes_analysed.csv", index=False)
print(analysed[["note_id","root_cause","confidence","validation_status"]])
print("API calls so far:", CALLS)

In [ ]:
expected = dict(zip([f"N{i:03d}" for i in range(1, 11)],
    ["forgot_or_no_reminder","distance_or_transport","cost_or_fee","other","illness_or_emergency",
     "work_or_schedule","forgot_or_no_reminder","distance_or_transport","other","work_or_schedule"]))
nonblank = analysed[analysed["validation_status"] != "skipped_blank"]
agree = sum(expected[r.note_id] == r.root_cause for r in nonblank.itertuples())
print(f"agreement with the answer key: {agree} of {len(nonblank)} non-blank notes (need >= 7 of 9)")

## Task 4.3 – break it on purpose

In [ ]:
bad_replies = {"confidence = 1.7": {"root_cause": "other", "confidence": 1.7, "summary": "Patient was unwell", "recommended_action": "Call back"},
               "missing recommended_action": {"root_cause": "other", "confidence": 0.5, "summary": "Patient was unwell"}}
for label, reply in bad_replies.items():
    try:
        FollowupInsight(**reply)
    except ValidationError as e:
        print(f"--- {label} ---\n{e}\n")

In [ ]:
rows = []
for nid in ["N007", "N009"]:
    text = notes.loc[notes["note_id"] == nid, "note_text"].iloc[0]
    for version, prompt in [("V1", f"Classify this note: {text}"), ("V2", craft_prompt(text))]:
        raw = call_model(prompt)
        try:
            rc = json.loads(raw).get("root_cause")
        except Exception:
            rc = None
        rows.append({"note": nid, "prompt": version, "root_cause": rc, "in_allowed_list": rc in ALLOWED})
        time.sleep(13)
print(pd.DataFrame(rows))
print("TOTAL API calls used in Tasks 4.1-4.3:", CALLS)

**Deploy:** V2 (CRAFT + allowed values + 'answer other rather than guess'), because it returns an allowed label for the Hinglish and ambiguous notes. (Confirm against your own table.)

## Task 4.4 – merge and cross-tabulate

In [ ]:
appts = pd.read_csv("appointments_clean.csv")
merged = analysed.merge(appts, on="appointment_id", how="left")
print(len(merged), "rows")
print(pd.crosstab(merged["root_cause"], merged["reminder_sent"]))
print("notes with reminder_sent = No:", (merged["reminder_sent"] == "No").sum(), "of", len(merged))